## 0. Импорт

In [96]:
import joblib
import numpy as np
import pandas as pd
import sklearn
from scipy.constants import precision
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

## 1. Предварительная обработка
1) Создайте тот же датафрейм, что и в предыдущем упражнении.
2) Используя train_test_splitпараметры test_size=0.2, random_state=21получите X_train, y_train, X_test, y_test. Используйте дополнительный параметр stratify.

In [80]:

df = pd.read_csv('../data/day-of-week-not-scaled.csv')
lastDf = pd.read_csv('../data/dayofweek.csv')
df = lastDf[['dayofweek'] + [c for c in df if c != 'dayofweek']]
df.head(3)


,dayofweek,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,4,-0.788667,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,4,-0.756764,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,4,-0.724861,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [81]:
X = df[df.drop(columns='dayofweek', axis=1).columns]
y = df['dayofweek']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. SVM
1) Используйте оптимальные параметры из предыдущего упражнения и обучите модель SVM.
2) Вам нужно рассчитать accuracy, precision, recall, ROC AUC.
precision и recall следует рассчитывать для каждого класса (используйте average='weighted').
3) ROC AUC Для каждого класса следует произвести расчет относительно любого другого класса (все возможные попарные комбинации), а затем для получения итоговой метрики следует применить взвешенное среднее.
4) Код в ячейке должен отображать результат, как показано ниже:
```accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878 

In [82]:
svc = SVC( C= 5, class_weight = None, gamma = 'scale', kernel = 'rbf', probability=True)
svc.fit(X_train, y_train)
y_pred = svc.predict(X_test)
y_pred_proba = svc.predict_proba(X_test)


In [83]:
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
rocAuc = roc_auc_score(y_test, y_pred_proba, multi_class='ovo', average='weighted')

print(f'precision is: {precision}')
print(f'recall is: {recall}')
print(f'rocAuc is: {rocAuc}')

precision is: 0.9073569397009559
recall is: 0.9053254437869822
rocAuc is: 0.9822859118639409


## 3. Дерево решений
Та же задача для дерева решений.

In [84]:
tree = DecisionTreeClassifier(class_weight = None, criterion = 'gini', max_depth = 16)
tree.fit(X_train, y_train)
y_pred = tree.predict(X_test)
y_pred_proba = tree.predict_proba(X_test)

In [85]:
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
rocAuc = roc_auc_score(y_test, y_pred_proba, multi_class='ovo', average='weighted')

print(f'precision is: {precision}')
print(f'recall is: {recall}')
print(f'rocAuc is: {rocAuc}')

precision is: 0.849806083571719
recall is: 0.8431952662721893
rocAuc is: 0.9307221662528181


## 4. Случайный лес
Та же задача для случайного леса.

In [86]:
rndForest = RandomForestClassifier(random_state=21, class_weight=None,
                                        criterion='entropy', n_estimators=100,
                                        n_jobs=-1, max_depth=14)

rndForest.fit(X_train, y_train)
y_pred = rndForest.predict(X_test)
y_pred_proba = rndForest.predict_proba(X_test)

In [87]:
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
rocAuc = roc_auc_score(y_test, y_pred_proba, multi_class='ovo', average='weighted')

print(f'precision is: {precision}')
print(f'recall is: {recall}')
print(f'rocAuc is: {rocAuc}')

precision is: 0.9136162461928916
recall is: 0.908284023668639
rocAuc is: 0.9846508961825863


## 5. Прогнозы
1) Выберите лучшую модель.
2) Проанализируйте: для каких weekday моделей ваша модель допускает наибольшее количество ошибок (в % от общего числа образцов этого класса в вашем полном наборе данных), для каких labname и для каких users.
3) Сохраните модель.

In [88]:
best_model = rndForest

df_test = X_test.copy()
df_test['true_label'] = y_test
df_test['pred_label'] = best_model.predict(X_test)

errors = df_test[df_test['true_label'] != df_test['pred_label']]
errors_count = errors['true_label'].value_counts()
total_count = df_test['true_label'].value_counts()

errors_percentage = (errors_count/total_count).sort_values(ascending=False)

users = [user for user in errors.columns if user.startswith('uid_user_')]
errors_labs = errors.drop(columns=['true_label', 'pred_label']+users).sum().sort_values(ascending=False)

labs = [lab for lab in errors.columns if lab.startswith('labname_') ]
errors_users = errors.drop(columns=['true_label', 'pred_label']+labs).sum().sort_values(ascending=False)

print("\nТоп классов с наибольшим % ошибок:")
print(errors_percentage.head())
print("\nТоп лабораторных с наибольшим % ошибок:")
print(errors_labs.head())
print("\nТоп пользователей с наибольшим % ошибок:")
print(errors_users.head())


Топ классов с наибольшим % ошибок:
true_label
0    0.259259
4    0.142857
1    0.127273
5    0.074074
6    0.070423
Name: count, dtype: float64

Топ лабораторных с наибольшим % ошибок:
labname_project1    14.0
labname_laba04       7.0
labname_laba04s      3.0
labname_code_rvw     1.0
labname_lab03        1.0
dtype: float64

Топ пользователей с наибольшим % ошибок:
uid_user_2     3.0
uid_user_4     3.0
uid_user_16    2.0
uid_user_30    2.0
uid_user_3     2.0
dtype: float64


In [99]:
joblib.dump(best_model, '../models/bestModel_rndForest.joblib')

['../models/bestModel_rndForest.joblib']

## 6. Функция
1) Напишите функцию, которая принимает список различных моделей и соответствующий список параметров (словари) и возвращает словарь, содержащий все 4 метрики для каждой модели.

In [93]:
def models_metrics(models, params, X_train, X_test, y_train, y_test):
    results = {}
    for model, params in zip(models, params):
        model.set_params(**params)
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        
        if hasattr(model, "predict_proba"):
            y_score = model.predict_proba(X_test)
        elif hasattr(model, "decision_function"):
            y_score = model.decision_function(X_test)
        else:
            # Если нет ни того, ни другого, ROC AUC посчитать нельзя
            y_score = None
        
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted')
        rec = recall_score(y_test, y_pred, average='weighted')
        
        if y_score is not None:
            roc = roc_auc_score(y_test, y_score, multi_class='ovo', average='weighted')
        else:
            roc = None
        results[model.__class__.__name__] = {
            'accuracy': acc,
            'precision': prec,
            'recall': rec,
            'roc_auc': roc
        }
    return results

In [94]:
models_to_test = [
    SVC(),
    DecisionTreeClassifier(),
    RandomForestClassifier()
]

params_to_test = [
    {'C': 5, 'kernel': 'rbf', 'gamma': 'scale', 'probability': True}, 
    {'max_depth': 16, 'criterion': 'gini'},                           
    {'n_estimators': 100, 'max_depth': 14, 'criterion': 'entropy'}    
]

metrics_dict = models_metrics(models_to_test, params_to_test, X_train,  X_test, y_train, y_test)

for model_name, metrics in metrics_dict.items():
    print(f"Model: {model_name}")
    for metric_name, value in metrics.items():
        print(f"  {metric_name}: {value:.5f}")
    print()

Model: SVC
  accuracy: 0.90533
  precision: 0.90736
  recall: 0.90533
  roc_auc: 0.98289

Model: DecisionTreeClassifier
  accuracy: 0.84320
  precision: 0.84760
  recall: 0.84320
  roc_auc: 0.93132

Model: RandomForestClassifier
  accuracy: 0.90828
  precision: 0.91261
  recall: 0.90828
  roc_auc: 0.98422

